# Robustness Analysis (1d U17 + 4h U12)

This notebook computes robustness for returns and utility-excess across costs and subperiods.
Artifacts are saved to `results/robustness` and `figures/robustness`.


In [1]:
# Cell 1: Setup + Config
from __future__ import annotations

import logging
from collections import OrderedDict
from pathlib import Path

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except Exception:
    display = print

from src.artifacts import safe_write_parquet
from src.metrics import compute_metrics_from_returns
from src.robustness import utility_penalized_return_worsen_dd

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
logger = logging.getLogger("robustness_nb")

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src").exists():
    raise FileNotFoundError(f"Cannot locate project root from cwd={Path.cwd()}")

RESULTS_DIR = PROJECT_ROOT / "results" / "robustness"
FIGURES_DIR = PROJECT_ROOT / "figures" / "robustness"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

INPUTS = {
    "base_1d_returns": PROJECT_ROOT / "results" / "runner_exp" / "1d" / "returns_oos.parquet",
    "base_1d_bench": PROJECT_ROOT / "results" / "runner_exp" / "1d" / "bench_returns_oos.parquet",
    "onchain_1d_returns": PROJECT_ROOT / "results" / "runner_onchain" / "returns_oos.parquet",
    "base_4h_returns": PROJECT_ROOT / "results" / "runner_exp" / "4h" / "returns_oos.parquet",
    "base_4h_bench": PROJECT_ROOT / "results" / "runner_exp" / "4h" / "bench_returns_oos.parquet",
}
for name, path in INPUTS.items():
    if not path.exists():
        raise FileNotFoundError(f"Missing required input: {name} -> {path}")

COSTS = ["Low", "Base", "High"]
LAMBDA = 1.0
PPY = {"1d": 365, "4h": 2190}
MIN_BARS = {"1d": 50, "4h": 300}

SUBPERIODS = OrderedDict({
    "COVID crash": ("2020-03-01", "2020-07-01"),
    "Bull 2021": ("2021-01-01", "2021-11-01"),
    "Bear 2022": ("2022-01-01", "2023-01-01"),
    "Recovery 2023": ("2023-01-01", "2024-01-01"),
    "Bull 2024": ("2024-01-01", "2025-01-01"),
    "H1 2025": ("2025-01-01", "2025-09-01"),
    "H2 2025+": ("2025-09-01", "2026-04-01"),
})

ASSET_EVENTS = {
    "XRP": {
        "Court victory Jul 2023": ("2023-07-01", "2023-10-01"),
    },
    "DOGE": {
        "Trump pump Nov 2024": ("2024-11-01", "2025-02-01"),
    },
    "BNB": {
        "SEC + CZ 2023": ("2023-06-01", "2023-12-01"),
    },
}

GROUP_RULES = OrderedDict({
    "Trend": ["T1:", "T2:", "T3:"],
    "Momentum": ["M1:"],
    "MeanRev": ["R1:", "R2:", "R3:"],
    "MeanRev+OC": ["R1OC:", "R2OC:", "R3OC:"],
    "Synergy": ["S1:", "S2:", "S3:", "S4:", "S5:"],
    "Synergy+OC": ["S1OC:", "S2OC:"],
})
GROUP_ORDER = list(GROUP_RULES.keys())

SYMBOL_MAP = {
    "BTC": "BTCUSDT",
    "ETH": "ETHUSDT",
    "XRP": "XRPUSDT",
    "DOGE": "DOGEUSDT",
    "BNB": "BNBUSDT",
}

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RESULTS_DIR:", RESULTS_DIR)
print("FIGURES_DIR:", FIGURES_DIR)


PROJECT_ROOT: c:\Users\prosh\OneDrive\Рабочий стол\git\diploma
RESULTS_DIR: c:\Users\prosh\OneDrive\Рабочий стол\git\diploma\results\robustness
FIGURES_DIR: c:\Users\prosh\OneDrive\Рабочий стол\git\diploma\figures\robustness


In [2]:
# Cell 2: Load + Universe Build

def ensure_date_col(df_in: pd.DataFrame) -> pd.DataFrame:
    df = df_in.copy()
    if "date" not in df.columns:
        if "datetime_utc" in df.columns:
            df = df.rename(columns={"datetime_utc": "date"})
        elif "index" in df.columns:
            df = df.rename(columns={"index": "date"})
        else:
            raise KeyError(f"No date-like column in dataframe. Columns: {list(df.columns)}")
    df["date"] = pd.to_datetime(df["date"])
    return df


def assert_unique_key(df: pd.DataFrame, *, name: str) -> None:
    key = ["date", "symbol", "cost", "strategy_id"]
    dups = int(df.duplicated(subset=key).sum())
    if dups > 0:
        raise RuntimeError(f"{name}: duplicate rows by {key}: {dups}")


def strategy_group(strategy_id: str) -> str:
    sid = str(strategy_id)
    for grp, prefixes in GROUP_RULES.items():
        if any(sid.startswith(p) for p in prefixes):
            return grp
    return "Other"


def symbols_for_returns(df: pd.DataFrame) -> list[str]:
    return sorted(df["symbol"].astype(str).unique().tolist())


base_1d_returns = ensure_date_col(pd.read_parquet(INPUTS["base_1d_returns"]))
base_1d_bench = ensure_date_col(pd.read_parquet(INPUTS["base_1d_bench"]))
onchain_1d_returns = ensure_date_col(pd.read_parquet(INPUTS["onchain_1d_returns"]))
base_4h_returns = ensure_date_col(pd.read_parquet(INPUTS["base_4h_returns"]))
base_4h_bench = ensure_date_col(pd.read_parquet(INPUTS["base_4h_bench"]))

assert_unique_key(base_1d_returns, name="base_1d_returns")
assert_unique_key(base_1d_bench, name="base_1d_bench")
assert_unique_key(onchain_1d_returns, name="onchain_1d_returns")
assert_unique_key(base_4h_returns, name="base_4h_returns")
assert_unique_key(base_4h_bench, name="base_4h_bench")

returns_1d_u17 = pd.concat([base_1d_returns, onchain_1d_returns], ignore_index=True)
assert_unique_key(returns_1d_u17, name="returns_1d_u17")
returns_4h_u12 = base_4h_returns.copy()

bench_1d = base_1d_bench.copy()
bench_4h = base_4h_bench.copy()

returns_1d_u17["timeframe"] = "1d"
returns_1d_u17["scenario"] = "u17"
returns_4h_u12["timeframe"] = "4h"
returns_4h_u12["scenario"] = "u12"

bench_1d["timeframe"] = "1d"
bench_1d["scenario"] = "u17"
bench_4h["timeframe"] = "4h"
bench_4h["scenario"] = "u12"

RETURNS_BY_KEY = {
    ("1d", "u17"): returns_1d_u17,
    ("4h", "u12"): returns_4h_u12,
}
BENCH_BY_KEY = {
    ("1d", "u17"): bench_1d,
    ("4h", "u12"): bench_4h,
}

rows = []
for key, df in RETURNS_BY_KEY.items():
    tf, sc = key
    rows.append({
        "timeframe": tf,
        "scenario": sc,
        "rows": len(df),
        "symbols": df["symbol"].astype(str).nunique(),
        "costs": df["cost"].astype(str).nunique(),
        "strategies": df["strategy_id"].astype(str).nunique(),
        "min_date": df["date"].min(),
        "max_date": df["date"].max(),
    })
summary_df = pd.DataFrame(rows)
display(summary_df)

u17_by_symbol = (
    returns_1d_u17.groupby("symbol")["strategy_id"].nunique().sort_index().rename("n_strategies")
)
print("1d U17 strategy counts by symbol:")
display(u17_by_symbol.reset_index())

if int(u17_by_symbol.get("BNBUSDT", 0)) != 12:
    raise AssertionError("Expected BNBUSDT to have 12 strategies in 1d U17")
for sym in ["BTCUSDT", "ETHUSDT", "XRPUSDT", "DOGEUSDT"]:
    if int(u17_by_symbol.get(sym, 0)) != 17:
        raise AssertionError(f"Expected {sym} to have 17 strategies in 1d U17")


,timeframe,scenario,rows,symbols,costs,strategies,min_date,max_date
0,1d,u17,401112,5,3,17,2020-09-16 00:00:00,2026-02-03
1,4h,u12,1894536,5,3,12,2020-08-22 04:00:00,2026-02-22


1d U17 strategy counts by symbol:


,symbol,n_strategies
0,BNBUSDT,12
1,BTCUSDT,17
2,DOGEUSDT,17
3,ETHUSDT,17
4,XRPUSDT,17


In [3]:
# Cell 3: Returns Cost Sensitivity (all strategies)

def _series_from_long(df: pd.DataFrame, *, symbol: str, cost: str, strategy_id: str) -> pd.Series:
    g = df[(df["symbol"] == symbol) & (df["cost"] == cost) & (df["strategy_id"] == strategy_id)]
    if g.empty:
        return pd.Series(dtype=float)
    g = g.sort_values("date")
    return g.set_index("date")["ret"].astype(float)


def stitched_returns_metrics(df_returns: pd.DataFrame, *, timeframe: str, scenario: str) -> pd.DataFrame:
    ppy = int(PPY[timeframe])
    min_bars = int(MIN_BARS[timeframe])
    out_rows = []
    symbols = sorted(df_returns["symbol"].astype(str).unique())
    strategies = sorted(df_returns["strategy_id"].astype(str).unique())

    for sym in symbols:
        for cost in COSTS:
            for sid in strategies:
                rs = _series_from_long(df_returns, symbol=sym, cost=cost, strategy_id=sid)
                if len(rs) < min_bars:
                    continue
                m = compute_metrics_from_returns(rs, periods_per_year=ppy)
                out_rows.append({
                    "timeframe": timeframe,
                    "scenario": scenario,
                    "symbol": sym,
                    "cost": cost,
                    "strategy_id": sid,
                    "strategy_group": strategy_group(sid),
                    "Total Return": float(m.get("Total Return", np.nan)),
                    "CAGR": float(m.get("CAGR", np.nan)),
                    "Ann. Vol": float(m.get("Ann. Vol", np.nan)),
                    "Sharpe": float(m.get("Sharpe", np.nan)),
                    "Sortino": float(m.get("Sortino", np.nan)),
                    "MaxDD": float(m.get("MaxDD", np.nan)),
                    "Calmar": float(m.get("Calmar", np.nan)),
                    "n_bars": int(m.get("n_bars", len(rs))),
                })
    return pd.DataFrame(out_rows)


returns_cost_all = pd.concat(
    [
        stitched_returns_metrics(RETURNS_BY_KEY[("1d", "u17")], timeframe="1d", scenario="u17"),
        stitched_returns_metrics(RETURNS_BY_KEY[("4h", "u12")], timeframe="4h", scenario="u12"),
    ],
    ignore_index=True,
)

safe_write_parquet(returns_cost_all, RESULTS_DIR / "robust_returns_cost_all.parquet")

compact_src = (
    returns_cost_all
    .groupby(["timeframe", "scenario", "symbol", "strategy_group", "cost"], as_index=False)["Calmar"]
    .mean()
)
compact_wide = compact_src.pivot_table(
    index=["timeframe", "scenario", "symbol", "strategy_group"],
    columns="cost",
    values="Calmar",
    aggfunc="mean",
).reset_index()
compact_wide.columns.name = None

for c in COSTS:
    if c not in compact_wide.columns:
        compact_wide[c] = np.nan

compact_wide["Calmar (Base)"] = compact_wide["Base"]
compact_wide["Calmar drop (High vs Base)"] = compact_wide["High"] - compact_wide["Base"]
compact_wide["Calmar drop pct (High vs Base)"] = np.where(
    compact_wide["Base"].abs() > 1e-12,
    (compact_wide["High"] - compact_wide["Base"]) / compact_wide["Base"].abs() * 100.0,
    np.nan,
)

robust_returns_cost_group_compact = compact_wide[
    [
        "timeframe", "scenario", "symbol", "strategy_group",
        "Low", "Base", "High",
        "Calmar (Base)", "Calmar drop (High vs Base)", "Calmar drop pct (High vs Base)",
    ]
].sort_values(["timeframe", "symbol", "strategy_group"])

safe_write_parquet(robust_returns_cost_group_compact, RESULTS_DIR / "robust_returns_cost_group_compact.parquet")

print("returns_cost_all:", returns_cost_all.shape)
print("robust_returns_cost_group_compact:", robust_returns_cost_group_compact.shape)
display(robust_returns_cost_group_compact.head(20))


returns_cost_all: (420, 14)
robust_returns_cost_group_compact: (48, 10)


,timeframe,scenario,symbol,strategy_group,Low,Base,High,Calmar (Base),Calmar drop (High vs Base),Calmar drop pct (High vs Base)
0,1d,u17,BNBUSDT,MeanRev,0.291592,0.287616,0.267984,0.287616,-0.019632,-6.825795
1,1d,u17,BNBUSDT,Momentum,1.673438,1.594101,1.439419,1.594101,-0.154682,-9.703404
2,1d,u17,BNBUSDT,Synergy,0.675167,0.667241,0.648676,0.667241,-0.018565,-2.782342
3,1d,u17,BNBUSDT,Trend,1.665076,1.646918,1.687968,1.646918,0.041050,2.492518
4,1d,u17,BTCUSDT,MeanRev,-0.009047,-0.016986,0.017031,-0.016986,0.034017,200.262225
5,1d,u17,BTCUSDT,MeanRev+OC,0.043923,0.037154,0.022864,0.037154,-0.014289,-38.460198
6,1d,u17,BTCUSDT,Momentum,0.816010,0.998636,0.952521,0.998636,-0.046114,-4.617744
7,1d,u17,BTCUSDT,Synergy,0.664905,0.655922,0.636749,0.655922,-0.019173,-2.923067
8,1d,u17,BTCUSDT,Synergy+OC,0.174765,0.167644,0.152255,0.167644,-0.015389,-9.179765
9,1d,u17,BTCUSDT,Trend,0.867224,0.854983,0.807324,0.854983,-0.047659,-5.574280


In [4]:
# Cell 4: Utility Cost Sensitivity (all strategies, lambda=1.0)

def utility_excess_cost_table(
    df_returns: pd.DataFrame,
    df_bench: pd.DataFrame,
    *,
    timeframe: str,
    scenario: str,
    lam: float,
) -> pd.DataFrame:
    min_bars = int(MIN_BARS[timeframe])
    out_rows = []
    symbols = sorted(df_returns["symbol"].astype(str).unique())
    strategies = sorted(df_returns["strategy_id"].astype(str).unique())

    for sym in symbols:
        for cost in COSTS:
            b = _series_from_long(df_bench, symbol=sym, cost=cost, strategy_id="BENCH:BuyHold")
            if b.empty:
                continue
            for sid in strategies:
                r = _series_from_long(df_returns, symbol=sym, cost=cost, strategy_id=sid)
                if r.empty:
                    continue
                r_al, b_al = r.align(b, join="inner")
                if len(r_al) < min_bars:
                    continue

                u = utility_penalized_return_worsen_dd(r_al, lam=float(lam))
                ub = utility_penalized_return_worsen_dd(b_al, lam=float(lam))
                ue = (u - ub)

                out_rows.append({
                    "timeframe": timeframe,
                    "scenario": scenario,
                    "symbol": sym,
                    "cost": cost,
                    "strategy_id": sid,
                    "strategy_group": strategy_group(sid),
                    "lam": float(lam),
                    "n_bars": int(len(ue)),
                    "utility_mean": float(u.mean()),
                    "bench_utility_mean": float(ub.mean()),
                    "utility_excess_mean": float(ue.mean()),
                })

    return pd.DataFrame(out_rows)


utility_cost_all = pd.concat(
    [
        utility_excess_cost_table(
            RETURNS_BY_KEY[("1d", "u17")],
            BENCH_BY_KEY[("1d", "u17")],
            timeframe="1d",
            scenario="u17",
            lam=LAMBDA,
        ),
        utility_excess_cost_table(
            RETURNS_BY_KEY[("4h", "u12")],
            BENCH_BY_KEY[("4h", "u12")],
            timeframe="4h",
            scenario="u12",
            lam=LAMBDA,
        ),
    ],
    ignore_index=True,
)

safe_write_parquet(utility_cost_all, RESULTS_DIR / f"robust_utility_cost_all_lam{LAMBDA:.1f}.parquet")

utility_compact_src = (
    utility_cost_all
    .groupby(["timeframe", "scenario", "symbol", "strategy_group", "cost"], as_index=False)["utility_excess_mean"]
    .mean()
)
utility_compact = utility_compact_src.pivot_table(
    index=["timeframe", "scenario", "symbol", "strategy_group"],
    columns="cost",
    values="utility_excess_mean",
    aggfunc="mean",
).reset_index()
utility_compact.columns.name = None

for c in COSTS:
    if c not in utility_compact.columns:
        utility_compact[c] = np.nan

utility_compact["UtilityExcess (Base)"] = utility_compact["Base"]
utility_compact["UtilityExcess drop (High vs Base)"] = utility_compact["High"] - utility_compact["Base"]
utility_compact["UtilityExcess drop pct (High vs Base)"] = np.where(
    utility_compact["Base"].abs() > 1e-12,
    (utility_compact["High"] - utility_compact["Base"]) / utility_compact["Base"].abs() * 100.0,
    np.nan,
)

robust_utility_cost_group_compact = utility_compact[
    [
        "timeframe", "scenario", "symbol", "strategy_group",
        "Low", "Base", "High",
        "UtilityExcess (Base)",
        "UtilityExcess drop (High vs Base)",
        "UtilityExcess drop pct (High vs Base)",
    ]
].sort_values(["timeframe", "symbol", "strategy_group"])

safe_write_parquet(
    robust_utility_cost_group_compact,
    RESULTS_DIR / f"robust_utility_cost_group_compact_lam{LAMBDA:.1f}.parquet",
)

print("utility_cost_all:", utility_cost_all.shape)
print("robust_utility_cost_group_compact:", robust_utility_cost_group_compact.shape)
display(robust_utility_cost_group_compact.head(20))


utility_cost_all: (420, 11)
robust_utility_cost_group_compact: (48, 10)


,timeframe,scenario,symbol,strategy_group,Low,Base,High,UtilityExcess (Base),UtilityExcess drop (High vs Base),UtilityExcess drop pct (High vs Base)
0,1d,u17,BNBUSDT,MeanRev,0.003270,0.003269,0.003250,0.003269,-0.000019,-0.579851
1,1d,u17,BNBUSDT,Momentum,0.001977,0.001963,0.001931,0.001963,-0.000032,-1.628588
2,1d,u17,BNBUSDT,Synergy,0.003810,0.003810,0.003809,0.003810,-0.000001,-0.028628
3,1d,u17,BNBUSDT,Trend,0.002551,0.002546,0.002476,0.002546,-0.000071,-2.780549
4,1d,u17,BTCUSDT,MeanRev,0.002987,0.003028,0.003130,0.003028,0.000103,3.387971
5,1d,u17,BTCUSDT,MeanRev+OC,0.003522,0.003519,0.003512,0.003519,-0.000007,-0.200359
6,1d,u17,BTCUSDT,Momentum,0.001817,0.001766,0.001739,0.001766,-0.000027,-1.530657
7,1d,u17,BTCUSDT,Synergy,0.003021,0.003018,0.003010,0.003018,-0.000007,-0.244329
8,1d,u17,BTCUSDT,Synergy+OC,0.004235,0.004231,0.004223,0.004231,-0.000008,-0.197704
9,1d,u17,BTCUSDT,Trend,0.002165,0.002164,0.002153,0.002164,-0.000010,-0.471520


In [5]:
# Cell 5: Returns Subperiods (global + events)

def subperiod_returns_table(
    df_returns: pd.DataFrame,
    *,
    timeframe: str,
    scenario: str,
    periods: dict[str, tuple[str, str]],
    period_type: str,
    symbols_filter: list[str] | None = None,
) -> pd.DataFrame:
    ppy = int(PPY[timeframe])
    min_bars = int(MIN_BARS[timeframe])
    out_rows = []

    symbols = sorted(df_returns["symbol"].astype(str).unique())
    if symbols_filter is not None:
        symbols = [s for s in symbols if s in set(symbols_filter)]

    strategies = sorted(df_returns["strategy_id"].astype(str).unique())

    for sym in symbols:
        for cost in COSTS:
            for sid in strategies:
                rs = _series_from_long(df_returns, symbol=sym, cost=cost, strategy_id=sid)
                if rs.empty:
                    continue
                for label, (start, end) in periods.items():
                    x = rs.loc[(rs.index >= pd.Timestamp(start)) & (rs.index < pd.Timestamp(end))]
                    if len(x) < min_bars:
                        continue
                    m = compute_metrics_from_returns(x, periods_per_year=ppy)
                    out_rows.append({
                        "timeframe": timeframe,
                        "scenario": scenario,
                        "period_type": period_type,
                        "period_label": label,
                        "period_start": pd.Timestamp(start),
                        "period_end": pd.Timestamp(end),
                        "symbol": sym,
                        "cost": cost,
                        "strategy_id": sid,
                        "strategy_group": strategy_group(sid),
                        "n_bars": int(m.get("n_bars", len(x))),
                        "Total Return": float(m.get("Total Return", np.nan)),
                        "CAGR": float(m.get("CAGR", np.nan)),
                        "Ann. Vol": float(m.get("Ann. Vol", np.nan)),
                        "Sharpe": float(m.get("Sharpe", np.nan)),
                        "Sortino": float(m.get("Sortino", np.nan)),
                        "MaxDD": float(m.get("MaxDD", np.nan)),
                        "Calmar": float(m.get("Calmar", np.nan)),
                    })
    return pd.DataFrame(out_rows)


returns_subperiods_all = pd.concat(
    [
        subperiod_returns_table(
            RETURNS_BY_KEY[("1d", "u17")],
            timeframe="1d",
            scenario="u17",
            periods=dict(SUBPERIODS),
            period_type="global",
        ),
        subperiod_returns_table(
            RETURNS_BY_KEY[("4h", "u12")],
            timeframe="4h",
            scenario="u12",
            periods=dict(SUBPERIODS),
            period_type="global",
        ),
    ],
    ignore_index=True,
)

asset_event_parts = []
for asset, events in ASSET_EVENTS.items():
    sym = SYMBOL_MAP.get(asset)
    if sym is None:
        continue
    asset_event_parts.append(
        subperiod_returns_table(
            RETURNS_BY_KEY[("1d", "u17")],
            timeframe="1d",
            scenario="u17",
            periods=events,
            period_type=f"event:{asset}",
            symbols_filter=[sym],
        )
    )
    asset_event_parts.append(
        subperiod_returns_table(
            RETURNS_BY_KEY[("4h", "u12")],
            timeframe="4h",
            scenario="u12",
            periods=events,
            period_type=f"event:{asset}",
            symbols_filter=[sym],
        )
    )

returns_asset_events_all = pd.concat(asset_event_parts, ignore_index=True) if asset_event_parts else pd.DataFrame()

safe_write_parquet(returns_subperiods_all, RESULTS_DIR / "robust_returns_subperiods_all.parquet")
safe_write_parquet(returns_asset_events_all, RESULTS_DIR / "robust_returns_asset_events_all.parquet")

print("returns_subperiods_all:", returns_subperiods_all.shape)
print("returns_asset_events_all:", returns_asset_events_all.shape)
display(returns_subperiods_all.head(10))
display(returns_asset_events_all.head(10))


returns_subperiods_all: (2331, 18)
returns_asset_events_all: (246, 18)


,timeframe,scenario,period_type,period_label,period_start,period_end,symbol,cost,strategy_id,strategy_group,n_bars,Total Return,CAGR,Ann. Vol,Sharpe,Sortino,MaxDD,Calmar
0,1d,u17,global,Bull 2021,2021-01-01,2021-11-01,BNBUSDT,Low,M1:TSMOM,Momentum,304,19.230430,35.988384,1.355982,3.267082,8.313275,-0.368938,97.545970
1,1d,u17,global,Bear 2022,2022-01-01,2023-01-01,BNBUSDT,Low,M1:TSMOM,Momentum,365,-0.347264,-0.347264,0.401299,-0.856197,-1.168942,-0.349033,-0.994933
2,1d,u17,global,Recovery 2023,2023-01-01,2024-01-01,BNBUSDT,Low,M1:TSMOM,Momentum,365,0.117140,0.117140,0.342979,0.494157,0.779984,-0.334232,0.350475
3,1d,u17,global,Bull 2024,2024-01-01,2025-01-01,BNBUSDT,Low,M1:TSMOM,Momentum,366,0.340870,0.339796,0.507750,0.824916,1.503928,-0.457464,0.742782
4,1d,u17,global,H1 2025,2025-01-01,2025-09-01,BNBUSDT,Low,M1:TSMOM,Momentum,243,0.097325,0.149703,0.363467,0.565716,0.887173,-0.230678,0.648969
5,1d,u17,global,H2 2025+,2025-09-01,2026-04-01,BNBUSDT,Low,M1:TSMOM,Momentum,96,0.048499,0.197295,0.703416,0.606789,1.016167,-0.367745,0.536499
6,1d,u17,global,Bull 2021,2021-01-01,2021-11-01,BNBUSDT,Low,R1:RSI_MR,MeanRev,304,-0.255574,-0.298381,0.820673,-0.004077,-0.005785,-0.567349,-0.525922
7,1d,u17,global,Bear 2022,2022-01-01,2023-01-01,BNBUSDT,Low,R1:RSI_MR,MeanRev,365,-0.167987,-0.167987,0.505644,-0.102576,-0.136815,-0.371500,-0.452187
8,1d,u17,global,Recovery 2023,2023-01-01,2024-01-01,BNBUSDT,Low,R1:RSI_MR,MeanRev,365,-0.029848,-0.029848,0.238862,-0.005383,-0.007133,-0.257561,-0.115888
9,1d,u17,global,Bull 2024,2024-01-01,2025-01-01,BNBUSDT,Low,R1:RSI_MR,MeanRev,366,0.186304,0.185751,0.236701,0.838613,1.239367,-0.201022,0.924034


,timeframe,scenario,period_type,period_label,period_start,period_end,symbol,cost,strategy_id,strategy_group,n_bars,Total Return,CAGR,Ann. Vol,Sharpe,Sortino,MaxDD,Calmar
0,1d,u17,event:XRP,Court victory Jul 2023,2023-07-01,2023-10-01,XRPUSDT,Low,M1:TSMOM,Momentum,92,-0.203043,-0.593600,0.359262,-2.317181,-2.821818,-0.210421,-2.821013
1,1d,u17,event:XRP,Court victory Jul 2023,2023-07-01,2023-10-01,XRPUSDT,Low,R1:RSI_MR,MeanRev,92,0.120784,0.572076,0.146567,3.161094,6.827841,-0.036590,15.634796
2,1d,u17,event:XRP,Court victory Jul 2023,2023-07-01,2023-10-01,XRPUSDT,Low,R1OC:RSI_MR_Onchain,MeanRev+OC,92,-0.111656,-0.374825,0.364471,-1.094516,-1.266515,-0.224047,-1.672972
3,1d,u17,event:XRP,Court victory Jul 2023,2023-07-01,2023-10-01,XRPUSDT,Low,R2:Bollinger_MR,MeanRev,92,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,1d,u17,event:XRP,Court victory Jul 2023,2023-07-01,2023-10-01,XRPUSDT,Low,R2OC:Bollinger_MR_Onchain,MeanRev+OC,92,0.391154,2.705322,1.499652,1.405543,6.563105,-0.262552,10.303929
5,1d,u17,event:XRP,Court victory Jul 2023,2023-07-01,2023-10-01,XRPUSDT,Low,R3:ZScore_MR,MeanRev,92,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
6,1d,u17,event:XRP,Court victory Jul 2023,2023-07-01,2023-10-01,XRPUSDT,Low,R3OC:ZScore_MR_Onchain,MeanRev+OC,92,-0.124498,-0.409919,0.339544,-1.370945,-1.546030,-0.213120,-1.923419
7,1d,u17,event:XRP,Court victory Jul 2023,2023-07-01,2023-10-01,XRPUSDT,Low,S1:MAFilter_RSI_MR,Synergy,92,0.529217,4.393377,1.495737,1.657796,7.875698,-0.224238,19.592508
8,1d,u17,event:XRP,Court victory Jul 2023,2023-07-01,2023-10-01,XRPUSDT,Low,S1OC:MAFilter_RSI_MR_Onchain,Synergy+OC,92,0.529217,4.393377,1.495737,1.657796,7.875698,-0.224238,19.592508
9,1d,u17,event:XRP,Court victory Jul 2023,2023-07-01,2023-10-01,XRPUSDT,Low,S2:MA200Filter_Bollinger_MR,Synergy,92,0.391154,2.705322,1.499652,1.405543,6.563105,-0.262552,10.303929


In [6]:
# Cell 6: Utility Subperiods (global + events, lambda=1.0)

def subperiod_utility_table(
    df_returns: pd.DataFrame,
    df_bench: pd.DataFrame,
    *,
    timeframe: str,
    scenario: str,
    periods: dict[str, tuple[str, str]],
    period_type: str,
    lam: float,
    symbols_filter: list[str] | None = None,
) -> pd.DataFrame:
    min_bars = int(MIN_BARS[timeframe])
    out_rows = []

    symbols = sorted(df_returns["symbol"].astype(str).unique())
    if symbols_filter is not None:
        symbols = [s for s in symbols if s in set(symbols_filter)]

    strategies = sorted(df_returns["strategy_id"].astype(str).unique())

    for sym in symbols:
        for cost in COSTS:
            b = _series_from_long(df_bench, symbol=sym, cost=cost, strategy_id="BENCH:BuyHold")
            if b.empty:
                continue
            for sid in strategies:
                r = _series_from_long(df_returns, symbol=sym, cost=cost, strategy_id=sid)
                if r.empty:
                    continue
                r_al, b_al = r.align(b, join="inner")
                if r_al.empty:
                    continue

                for label, (start, end) in periods.items():
                    mask = (r_al.index >= pd.Timestamp(start)) & (r_al.index < pd.Timestamp(end))
                    rs = r_al.loc[mask]
                    bs = b_al.loc[mask]
                    if len(rs) < min_bars:
                        continue

                    u = utility_penalized_return_worsen_dd(rs, lam=float(lam))
                    ub = utility_penalized_return_worsen_dd(bs, lam=float(lam))
                    ue = (u - ub)

                    out_rows.append({
                        "timeframe": timeframe,
                        "scenario": scenario,
                        "period_type": period_type,
                        "period_label": label,
                        "period_start": pd.Timestamp(start),
                        "period_end": pd.Timestamp(end),
                        "symbol": sym,
                        "cost": cost,
                        "strategy_id": sid,
                        "strategy_group": strategy_group(sid),
                        "lam": float(lam),
                        "n_bars": int(len(ue)),
                        "utility_mean": float(u.mean()),
                        "bench_utility_mean": float(ub.mean()),
                        "utility_excess_mean": float(ue.mean()),
                    })

    return pd.DataFrame(out_rows)


utility_subperiods_all = pd.concat(
    [
        subperiod_utility_table(
            RETURNS_BY_KEY[("1d", "u17")],
            BENCH_BY_KEY[("1d", "u17")],
            timeframe="1d",
            scenario="u17",
            periods=dict(SUBPERIODS),
            period_type="global",
            lam=LAMBDA,
        ),
        subperiod_utility_table(
            RETURNS_BY_KEY[("4h", "u12")],
            BENCH_BY_KEY[("4h", "u12")],
            timeframe="4h",
            scenario="u12",
            periods=dict(SUBPERIODS),
            period_type="global",
            lam=LAMBDA,
        ),
    ],
    ignore_index=True,
)

utility_event_parts = []
for asset, events in ASSET_EVENTS.items():
    sym = SYMBOL_MAP.get(asset)
    if sym is None:
        continue
    utility_event_parts.append(
        subperiod_utility_table(
            RETURNS_BY_KEY[("1d", "u17")],
            BENCH_BY_KEY[("1d", "u17")],
            timeframe="1d",
            scenario="u17",
            periods=events,
            period_type=f"event:{asset}",
            lam=LAMBDA,
            symbols_filter=[sym],
        )
    )
    utility_event_parts.append(
        subperiod_utility_table(
            RETURNS_BY_KEY[("4h", "u12")],
            BENCH_BY_KEY[("4h", "u12")],
            timeframe="4h",
            scenario="u12",
            periods=events,
            period_type=f"event:{asset}",
            lam=LAMBDA,
            symbols_filter=[sym],
        )
    )

utility_asset_events_all = pd.concat(utility_event_parts, ignore_index=True) if utility_event_parts else pd.DataFrame()

safe_write_parquet(utility_subperiods_all, RESULTS_DIR / f"robust_utility_subperiods_all_lam{LAMBDA:.1f}.parquet")
safe_write_parquet(utility_asset_events_all, RESULTS_DIR / f"robust_utility_asset_events_all_lam{LAMBDA:.1f}.parquet")

print("utility_subperiods_all:", utility_subperiods_all.shape)
print("utility_asset_events_all:", utility_asset_events_all.shape)
display(utility_subperiods_all.head(10))
display(utility_asset_events_all.head(10))


utility_subperiods_all: (2331, 15)
utility_asset_events_all: (246, 15)


,timeframe,scenario,period_type,period_label,period_start,period_end,symbol,cost,strategy_id,strategy_group,lam,n_bars,utility_mean,bench_utility_mean,utility_excess_mean
0,1d,u17,global,Bull 2021,2021-01-01,2021-11-01,BNBUSDT,Low,M1:TSMOM,Momentum,1.0,304,0.001044,-0.002789,3.832878e-03
1,1d,u17,global,Bear 2022,2022-01-01,2023-01-01,BNBUSDT,Low,M1:TSMOM,Momentum,1.0,365,-0.005463,-0.010206,4.742736e-03
2,1d,u17,global,Recovery 2023,2023-01-01,2024-01-01,BNBUSDT,Low,M1:TSMOM,Momentum,1.0,365,-0.003049,-0.004925,1.875584e-03
3,1d,u17,global,Bull 2024,2024-01-01,2025-01-01,BNBUSDT,Low,M1:TSMOM,Momentum,1.0,366,-0.004362,-0.005646,1.283838e-03
4,1d,u17,global,H1 2025,2025-01-01,2025-09-01,BNBUSDT,Low,M1:TSMOM,Momentum,1.0,243,-0.004254,-0.005751,1.497428e-03
5,1d,u17,global,H2 2025+,2025-09-01,2026-04-01,BNBUSDT,Low,M1:TSMOM,Momentum,1.0,96,-0.009358,-0.009358,-4.049527e-13
6,1d,u17,global,Bull 2021,2021-01-01,2021-11-01,BNBUSDT,Low,R1:RSI_MR,MeanRev,1.0,304,-0.004976,-0.002789,-2.186982e-03
7,1d,u17,global,Bear 2022,2022-01-01,2023-01-01,BNBUSDT,Low,R1:RSI_MR,MeanRev,1.0,365,-0.004715,-0.010206,5.491377e-03
8,1d,u17,global,Recovery 2023,2023-01-01,2024-01-01,BNBUSDT,Low,R1:RSI_MR,MeanRev,1.0,365,-0.001712,-0.004925,3.212653e-03
9,1d,u17,global,Bull 2024,2024-01-01,2025-01-01,BNBUSDT,Low,R1:RSI_MR,MeanRev,1.0,366,-0.000978,-0.005646,4.667564e-03


,timeframe,scenario,period_type,period_label,period_start,period_end,symbol,cost,strategy_id,strategy_group,lam,n_bars,utility_mean,bench_utility_mean,utility_excess_mean
0,1d,u17,event:XRP,Court victory Jul 2023,2023-07-01,2023-10-01,XRPUSDT,Low,M1:TSMOM,Momentum,1.0,92,-0.006638,-0.005914,-0.000724
1,1d,u17,event:XRP,Court victory Jul 2023,2023-07-01,2023-10-01,XRPUSDT,Low,R1:RSI_MR,MeanRev,1.0,92,0.000593,-0.005914,0.006507
2,1d,u17,event:XRP,Court victory Jul 2023,2023-07-01,2023-10-01,XRPUSDT,Low,R1OC:RSI_MR_Onchain,MeanRev+OC,1.0,92,-0.005130,-0.005914,0.000784
3,1d,u17,event:XRP,Court victory Jul 2023,2023-07-01,2023-10-01,XRPUSDT,Low,R2:Bollinger_MR,MeanRev,1.0,92,0.000000,-0.005914,0.005914
4,1d,u17,event:XRP,Court victory Jul 2023,2023-07-01,2023-10-01,XRPUSDT,Low,R2OC:Bollinger_MR_Onchain,MeanRev+OC,1.0,92,0.000877,-0.005914,0.006791
5,1d,u17,event:XRP,Court victory Jul 2023,2023-07-01,2023-10-01,XRPUSDT,Low,R3:ZScore_MR,MeanRev,1.0,92,0.000000,-0.005914,0.005914
6,1d,u17,event:XRP,Court victory Jul 2023,2023-07-01,2023-10-01,XRPUSDT,Low,R3OC:ZScore_MR_Onchain,MeanRev+OC,1.0,92,-0.004562,-0.005914,0.001352
7,1d,u17,event:XRP,Court victory Jul 2023,2023-07-01,2023-10-01,XRPUSDT,Low,S1:MAFilter_RSI_MR,Synergy,1.0,92,0.002598,-0.005914,0.008512
8,1d,u17,event:XRP,Court victory Jul 2023,2023-07-01,2023-10-01,XRPUSDT,Low,S1OC:MAFilter_RSI_MR_Onchain,Synergy+OC,1.0,92,0.002598,-0.005914,0.008512
9,1d,u17,event:XRP,Court victory Jul 2023,2023-07-01,2023-10-01,XRPUSDT,Low,S2:MA200Filter_Bollinger_MR,Synergy,1.0,92,0.000877,-0.005914,0.006791


In [7]:
# Cell 7: Visual Views (heatmaps)
import matplotlib.pyplot as plt


def _ordered_group_values(df: pd.DataFrame) -> list[str]:
    present = set(df["strategy_group"].astype(str).unique().tolist())
    ordered = [g for g in GROUP_ORDER if g in present]
    rest = sorted([g for g in present if g not in set(GROUP_ORDER)])
    return ordered + rest


def save_group_heatmap(
    pivot_df: pd.DataFrame,
    *,
    title: str,
    out_path: Path,
    cmap_name: str,
    value_fmt: str = "{:.2f}",
) -> None:
    if pivot_df.empty:
        return

    rows = list(pivot_df.index)
    cols = list(pivot_df.columns)

    data = pivot_df.to_numpy(dtype=float)
    masked = np.ma.masked_invalid(data)

    cmap = plt.get_cmap(cmap_name).copy()
    cmap.set_bad(color="#d9d9d9")

    fig_w = max(8, 1.3 * len(cols) + 2)
    fig_h = max(4, 0.55 * len(rows) + 2)
    fig, ax = plt.subplots(figsize=(fig_w, fig_h))
    im = ax.imshow(masked, aspect="auto", cmap=cmap)

    ax.set_xticks(np.arange(len(cols)))
    ax.set_yticks(np.arange(len(rows)))
    ax.set_xticklabels(cols, rotation=45, ha="right")
    ax.set_yticklabels(rows)
    ax.set_title(title)

    for i in range(len(rows)):
        for j in range(len(cols)):
            v = data[i, j]
            if np.isfinite(v):
                ax.text(j, i, value_fmt.format(v), ha="center", va="center", fontsize=8)

    fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
    fig.tight_layout()
    out_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_path, dpi=140, bbox_inches="tight")
    plt.close(fig)


returns_heatmap_files = []
utility_heatmap_files = []

for timeframe, scenario in [("1d", "u17"), ("4h", "u12")]:
    symbols = sorted(
        returns_subperiods_all[
            (returns_subperiods_all["timeframe"] == timeframe)
            & (returns_subperiods_all["scenario"] == scenario)
        ]["symbol"].astype(str).unique().tolist()
    )

    for sym in symbols:
        g_ret = returns_subperiods_all[
            (returns_subperiods_all["timeframe"] == timeframe)
            & (returns_subperiods_all["scenario"] == scenario)
            & (returns_subperiods_all["symbol"] == sym)
            & (returns_subperiods_all["cost"] == "Base")
        ].copy()
        if not g_ret.empty:
            grp = (
                g_ret.groupby(["strategy_group", "period_label"], as_index=False)["Calmar"]
                .mean()
            )
            col_order = list(SUBPERIODS.keys())
            pivot = grp.pivot(index="strategy_group", columns="period_label", values="Calmar")
            pivot = pivot.reindex(index=_ordered_group_values(grp), columns=col_order)

            out = FIGURES_DIR / f"heatmap_returns_{timeframe}_{scenario}_{sym}_base.png"
            save_group_heatmap(
                pivot,
                title=f"Returns Calmar Heatmap | {timeframe} {scenario} | {sym} | Base",
                out_path=out,
                cmap_name="RdYlGn",
            )
            returns_heatmap_files.append(str(out))

        g_ut = utility_subperiods_all[
            (utility_subperiods_all["timeframe"] == timeframe)
            & (utility_subperiods_all["scenario"] == scenario)
            & (utility_subperiods_all["symbol"] == sym)
            & (utility_subperiods_all["cost"] == "Base")
            & (utility_subperiods_all["lam"] == float(LAMBDA))
        ].copy()
        if not g_ut.empty:
            grp_u = (
                g_ut.groupby(["strategy_group", "period_label"], as_index=False)["utility_excess_mean"]
                .mean()
            )
            col_order = list(SUBPERIODS.keys())
            pivot_u = grp_u.pivot(index="strategy_group", columns="period_label", values="utility_excess_mean")
            pivot_u = pivot_u.reindex(index=_ordered_group_values(grp_u), columns=col_order)

            out_u = FIGURES_DIR / f"heatmap_utility_lam{LAMBDA:.1f}_{timeframe}_{scenario}_{sym}_base.png"
            save_group_heatmap(
                pivot_u,
                title=f"Utility Excess Heatmap (lam={LAMBDA:.1f}) | {timeframe} {scenario} | {sym} | Base",
                out_path=out_u,
                cmap_name="RdYlGn",
                value_fmt="{:.4f}",
            )
            utility_heatmap_files.append(str(out_u))

fig_inventory = pd.DataFrame(
    {
        "type": ["returns"] * len(returns_heatmap_files) + ["utility"] * len(utility_heatmap_files),
        "path": returns_heatmap_files + utility_heatmap_files,
    }
)
safe_write_parquet(fig_inventory, RESULTS_DIR / "heatmap_inventory.parquet")

print("returns heatmaps:", len(returns_heatmap_files))
print("utility heatmaps:", len(utility_heatmap_files))
display(fig_inventory.head(20))


returns heatmaps: 10
utility heatmaps: 10


,type,path
0,returns,c:\Users\prosh\OneDrive\Рабочий стол\git\diplo...
1,returns,c:\Users\prosh\OneDrive\Рабочий стол\git\diplo...
2,returns,c:\Users\prosh\OneDrive\Рабочий стол\git\diplo...
3,returns,c:\Users\prosh\OneDrive\Рабочий стол\git\diplo...
4,returns,c:\Users\prosh\OneDrive\Рабочий стол\git\diplo...
5,returns,c:\Users\prosh\OneDrive\Рабочий стол\git\diplo...
6,returns,c:\Users\prosh\OneDrive\Рабочий стол\git\diplo...
7,returns,c:\Users\prosh\OneDrive\Рабочий стол\git\diplo...
8,returns,c:\Users\prosh\OneDrive\Рабочий стол\git\diplo...
9,returns,c:\Users\prosh\OneDrive\Рабочий стол\git\diplo...
